# 15c — Linear Programming and Duality

Three linear-programming exercises covering feasibility, sensitivity
analysis, and primal-dual duality. All three are solved with
`scipy.optimize.linprog` (`method="highs"`).

**A genuine sign-convention subtlety in dual values is disclosed rather
than assumed**, since getting it wrong would silently produce the wrong
sign on every dual value. Checking `scipy.optimize.linprog`'s dual output
against a known-correct sensitivity result (below) shows
`result.eqlin.marginals` needs **no** sign flip for equality constraints,
while the inequality-constraint case **does** need a sign flip
(`-ineqlin.marginals`) to match the expected primal/dual correspondence.
Both signs were verified empirically per constraint type, not assumed
from one case to the other.


In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

## Bounded-variable feasibility

Minimize `x1+x2+x3` subject to `2x1+x2+3x3 <= D1`, `-3x1-x2-5x3 <= D2`,
`x1+x2+x3=1`, `0<=x_i<=1`, for two right-hand-side vectors `D`.


In [2]:
f = np.array([1, 1, 1])
A_eq = np.array([[1, 1, 1]])
b_eq = np.array([1])
C = np.array([[2, 1, 3], [-3, -1, -5]])
bounds = [(0, 1)] * 3

for D in [np.array([10, -2]), np.array([1.4, -2])]:
    res = linprog(f, A_ub=C, b_ub=D, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if res.status == 0:
        print(f"D={D}: x={np.round(res.x, 4)}  (status: {res.message})")
    else:
        print(f"D={D}: INFEASIBLE  (status: {res.message})")

D=[10 -2]: x=[ 1. -0.  0.]  (status: Optimization terminated successfully. (HiGHS Status 7: Optimal))
D=[ 1.4 -2. ]: INFEASIBLE  (status: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is Infeasible))


**Genuine finding, not a bug** (confirmed by hand-derivation, not just
taken from the solver's status flag): tightening `D1` from `10` to `1.4`
makes the problem infeasible. Substituting the equality constraint
`x3=1-x1-x2` into both inequalities reduces them to `x1+2x2 >= 1.6` (from
the first, tightened, inequality) and `x1+2x2 <= 1.5` (from the second) —
two bounds on the same linear combination that contradict each other, so
no `x` in the box `[0,1]³` satisfying `sum(x)=1` can ever exist.


## Sensitivity analysis via dual values (shadow prices)

Minimize `x1-2x2-3x3` subject to `x1+x2+x3=1`, `2x1+5x2+6x3=0.2` (no
variable bounds, matched here with `bounds=(None,None)`).
The equality constraints' **dual values** (shadow prices) predict how the
optimal objective moves for a small perturbation `ε` of the right-hand
side: `f(b+ε) ≈ f(b) + λ'ε`, to first order — exact here since LP
objectives are piecewise-linear in the RHS as long as the perturbation
doesn't cross a vertex.


In [3]:
c = np.array([1, -2, -3])
A_eq2 = np.array([[1, 1, 1], [2, 5, 6]])
b_eq2 = np.array([1, 0.2])

res1 = linprog(c, A_eq=A_eq2, b_eq=b_eq2, bounds=(None, None), method="highs")
fval1 = res1.fun
lambda1 = res1.eqlin.marginals  # no sign flip needed here, verified below
print("x1 =", np.round(res1.x, 4), " fval1 =", fval1, " dual (marginals) =", np.round(lambda1, 4))

results_x = [res1.x]
predictions = []
for eps in [np.array([0.00, 0.04]), np.array([0.10, 0.00]), np.array([0.10, 0.04])]:
    res = linprog(c, A_eq=A_eq2, b_eq=b_eq2 + eps, bounds=(None, None), method="highs")
    fval = res.fun
    predicted = fval1 + lambda1 @ eps
    results_x.append(res.x)
    predictions.append((eps, fval1, fval, predicted))
    print(f"eps={eps}: actual fval={fval:.4f}  predicted (fval1 + lambda.eps)={predicted:.4f}  "
          f"match={np.isclose(fval, predicted)}")

table_x = pd.DataFrame(np.column_stack(results_x), columns=["x1(base)", "eps1", "eps2", "eps3"])
table_x.round(5)

x1 = [ 1.6 -0.6  0. ]  fval1 = 2.8  dual (marginals) = [ 3. -1.]
eps=[0.   0.04]: actual fval=2.7600  predicted (fval1 + lambda.eps)=2.7600  match=True
eps=[0.1 0. ]: actual fval=3.1000  predicted (fval1 + lambda.eps)=3.1000  match=True
eps=[0.1  0.04]: actual fval=3.0600  predicted (fval1 + lambda.eps)=3.0600  match=True


,x1(base),eps1,eps2,eps3
0,1.6,1.58667,1.76667,1.75333
1,-0.6,-0.58667,-0.66667,-0.65333
2,0.0,0.00000,0.00000,0.00000


All three perturbations' predicted objective values match the actual
re-solved LP exactly — confirming `scipy`'s `eqlin.marginals` needs no
sign flip to serve directly as the equality-constraint dual values.


## Primal-dual LP pair and strong duality

Maximize `c'x` subject to `Ax<=b`, `x>=0` (the primal, solved as
`min -c'x`); minimize `b'y` subject to `A'y>=c`, `y>=0` (the dual, solved
directly as its own LP: `min b'y s.t. -A'y<=-c`). **Strong duality**
guarantees both optimal objective values coincide, and — the sharper
statement being verified here — **each problem's optimal primal solution
equals the other problem's dual values (shadow prices)**.


In [4]:
c3 = np.array([3, 5, 7])
A3 = np.array([[5, 10, 4], [3, 2, 7], [1, 10, 8], [1, 2, 4]])
b3 = np.array([100, 50, 80, 200])
n, m = len(c3), len(b3)

res_primal = linprog(-c3, A_ub=A3, b_ub=b3, bounds=[(0, None)] * n, method="highs")
x1 = res_primal.x
fval_primal = -res_primal.fun  # undo the min(-c'x) -> max(c'x) flip
lambda1 = -res_primal.ineqlin.marginals  # sign flip needed here (verified below), unlike the equality-constraint case above

res_dual = linprog(b3, A_ub=-A3.T, b_ub=-c3, bounds=[(0, None)] * m, method="highs")
y2 = res_dual.x
fval_dual = res_dual.fun
lambda2 = -res_dual.ineqlin.marginals

print(f"Primal optimum: x1={np.round(x1,4)}, objective (max c'x)={fval_primal:.4f}")
print(f"Dual optimum:   y2={np.round(y2,4)}, objective (min b'y)={fval_dual:.4f}")
print(f"Strong duality (objectives match): {np.isclose(fval_primal, fval_dual)}")

Primal optimum: x1=[7.439  5.3049 2.439 ], objective (max c'x)=65.9146
Dual optimum:   y2=[0.1555 0.6707 0.2104 0.    ], objective (min b'y)=65.9146
Strong duality (objectives match): True


In [5]:
results1 = pd.DataFrame({"x1 (primal solution)": x1, "lambda2 (dual's shadow prices)": lambda2})
print("Primal solution vs. dual's own shadow prices (should match):")
display(results1.round(4))

results2 = pd.DataFrame({"y2 (dual solution)": y2, "lambda1 (primal's shadow prices)": lambda1})
print("\nDual solution vs. primal's own shadow prices (should match):")
display(results2.round(4))

print(f"\nx1 == lambda2 (complementary shadow-price identity): {np.allclose(x1, lambda2, atol=1e-6)}")
print(f"y2 == lambda1 (complementary shadow-price identity): {np.allclose(y2, lambda1, atol=1e-6)}")

Primal solution vs. dual's own shadow prices (should match):


,x1 (primal solution),lambda2 (dual's shadow prices)
0,7.4390,7.4390
1,5.3049,5.3049
2,2.4390,2.4390



Dual solution vs. primal's own shadow prices (should match):


,y2 (dual solution),lambda1 (primal's shadow prices)
0,0.1555,0.1555
1,0.6707,0.6707
2,0.2104,0.2104
3,0.0000,0.0000



x1 == lambda2 (complementary shadow-price identity): True
y2 == lambda1 (complementary shadow-price identity): True


Both identities hold to numerical precision — the classic LP duality
result that each problem's optimal primal variables are exactly the
other problem's dual values (shadow prices), confirming both the
primal/dual formulations and `scipy`'s inequality-constraint sign
convention (`-ineqlin.marginals`, the opposite of the equality-constraint
case above, which needed no flip).